# Train the YOLO detection models (crop + RVIP) — baseline

Drop-in replacement for the two nnUNet **detection** stages, keeping the rest of the
pipeline (upstream preprocessing + LV segmentation) unchanged so it is a fair comparison:

- **CROP model** (Dataset110) — step-1 whole-heart ROI in the full image.
- **RVIP model** (Dataset105) — the 2 RV insertion points, from the IPs-only mask (`IP1 = 1` anterior, `IP2 = 2` inferior; no LV).

The LV segmentation stays nnUNet (Dataset100, trained separately). This notebook only
**trains** the two detectors and sanity-checks the labels; wiring YOLO into
`inference.ipynb` is a separate step.

**Baseline = first contrast only.** YOLO reads the *same* `imagesTr` NIfTIs nnUNet trains
on (the output of the existing preprocessing) and uses the first contrast `_0000`
(`CHANNELS = (0, 0, 0)`, replicated to R/G/B — YOLO needs 8-bit RGB). The multi-contrast
extension `(0, 1, 2)` stays available for a later study but is **not** the baseline.

**Test data is off-limits:** only `imagesTr`/`labelsTr` are read here. `imagesTs`/`labelsTs`
are never touched.

**No manual labelling** — labels are derived from the existing nnUNet ground truth by `yolo_pipeline.py`.

In [ ]:
import os, glob, json, sys
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# yolo_pipeline.py lives in the repo root next to this notebook
sys.path.append(os.getcwd())
import yolo_pipeline as yp

# YOLO backend (ultralytics is only needed for training/inference, not for the export)
try:
    import ultralytics
    print('ultralytics', ultralytics.__version__)
except ImportError:
    print('ultralytics not installed ->  pip install ultralytics')

## 1. Config
Point these at your ground-truth nnUNet datasets and pick hyperparameters.
Everything below is derived from these.

In [ ]:
# =====================================================================
#  RVIP model  ->  train on Dataset105 (combined LV + insertion points)
# =====================================================================
# imagesTr holds the contrasts (*_0000 first). Baseline uses ONLY _0000.
# CONFIRM label ids against dataset.json (see RVIP_LABEL_IDS below).
RVIP_IMAGES = '/home/sastocke/nnUNet/nnUNet_raw/Dataset105_HannumSmartHealthDataIPs/imagesTr'
RVIP_LABELS = '/home/sastocke/nnUNet/nnUNet_raw/Dataset105_HannumSmartHealthDataIPs/labelsTr'
RVIP_SPLITS = '/home/sastocke/nnUNet/nnUNet_preprocessed/Dataset105_HannumSmartHealthDataIPs/splits_final.json'  # train/val; or None

# Insertion-point label ids, straight from Dataset105 dataset.json (IPs-only, no LV):
#   IP1 = 1 (anterior insertion point), IP2 = 2 (inferior insertion point).
RVIP_ANTERIOR_LABEL = 1   # IP1
RVIP_INFERIOR_LABEL  = 2   # IP2

# =====================================================================
#  CROP model  ->  train on Dataset110 (whole-heart box, single contrast)
# =====================================================================
# Step-1 ROI dataset: whole-heart binary mask (Heart=1). dataset.json declares 4
# contrasts (_0000.._0003); the baseline uses _0000 only (CROP_CHANNELS below).
CROP_IMAGES = '/home/sastocke/nnUNet/nnUNet_raw/Dataset110_HannumSmarthHealthDataCrop/imagesTr'
CROP_LABELS = '/home/sastocke/nnUNet/nnUNet_raw/Dataset110_HannumSmarthHealthDataCrop/labelsTr'
CROP_SPLITS = '/home/sastocke/nnUNet/nnUNet_preprocessed/Dataset110_HannumSmarthHealthDataCrop/splits_final.json'  # or None

# NOTE: imagesTs/labelsTs are the TEST set -> intentionally NOT referenced anywhere.
#       Training uses imagesTr/labelsTr only, split into train/val via splits_final.json.

OUT_ROOT = '/home/sastocke/nnUNet/yolo_datasets'   # YOLO-format data + runs go here
FOLD     = 2                                        # which split fold to mirror

# ---------- channel selection (nnUNet contrast index -> YOLO R/G/B) ----------
# Baseline: first contrast only, replicated to R/G/B (YOLO needs 8-bit RGB).
#   (0, 0, 0) -> use _0000 only  [BASELINE for both models]
#   (0, 1, 2) -> three distinct contrasts  [multi-contrast extension; RVIP only]
CHANNELS      = (0, 0, 0)    # RVIP (Dataset105): first-contrast baseline
CROP_CHANNELS = (0, 0, 0)    # CROP (Dataset110): single contrast

# ---------- hyperparameters ----------
RVIP_IMGSZ   = 256          # YOLO letterboxes to this (multiple of 32)
CROP_IMGSZ   = 256
RVIP_BOX_PX  = 10           # px size (in the 256 frame) of the box drawn around each insertion point
EPOCHS       = 100
YOLO_WEIGHTS = 'yolov8n.pt'  # nano is plenty for single-structure / 2-point detection

assert len(CHANNELS) == 3 and len(CROP_CHANNELS) == 3, 'CHANNELS must have 3 entries (R, G, B).'
tag = lambda ch: '(multi-contrast extension)' if len(set(ch)) > 1 else '(first-contrast baseline)'
print('RVIP CHANNELS =', CHANNELS, tag(CHANNELS))
print('CROP CHANNELS =', CROP_CHANNELS, tag(CROP_CHANNELS))

## 2. Helper functions
`build_train_val` pairs `imagesTr` cases with `labelsTr` and assigns train/val from the
nnUNet split (test set is never touched); `show_yolo` overlays the exported boxes so you
can eyeball that the labels are correct.

In [ ]:
def load_split_map(splits_json, fold=0):
    """Return {case_id: 'train'|'val'} from an nnUNet splits_final.json."""
    with open(splits_json) as f:
        splits = json.load(f)
    m = {c: 'train' for c in splits[fold]['train']}
    m.update({c: 'val' for c in splits[fold]['val']})
    return m

def build_samples(images_dir, labels_dir, split_map=None, channel='_0000', default_split='train'):
    """Pair imagesXx channel-0 files with their labels; attach a split label.

    Only the *_0000 file is listed here; yolo_pipeline resolves the other
    contrasts (_0001/_0002/...) from CHANNELS at export/inference time.
    """
    samples = []
    for img in sorted(glob.glob(os.path.join(images_dir, f'*{channel}.nii.gz'))):
        case = os.path.basename(img).replace(f'{channel}.nii.gz', '')
        lbl  = os.path.join(labels_dir, case + '.nii.gz')
        if not os.path.exists(lbl):
            continue
        split = split_map.get(case, default_split) if split_map else default_split
        samples.append({'image_path': img, 'mask_path': lbl, 'name': case, 'split': split})
    return samples

def build_train_val(images_tr, labels_tr, splits_json=None, fold=0):
    """Pair imagesTr with labelsTr and split train/val. Test set is never read."""
    split_map = load_split_map(splits_json, fold) if splits_json else None
    return build_samples(images_tr, labels_tr, split_map)

def show_yolo(img_dir, lbl_dir, n=3, title=''):
    """Overlay exported YOLO boxes on their images to verify labels look right."""
    from PIL import Image
    for p in sorted(glob.glob(os.path.join(img_dir, '*.png')))[:n]:
        im = np.array(Image.open(p)); H, W = im.shape[:2]
        txt = os.path.join(lbl_dir, os.path.basename(p).replace('.png', '.txt'))
        plt.figure(figsize=(4, 4)); plt.imshow(im)   # RGB PNG (first contrast, replicated)
        if os.path.exists(txt):
            for line in open(txt):
                cls, cx, cy, w, h = map(float, line.split())
                plt.gca().add_patch(plt.Rectangle(((cx-w/2)*W, (cy-h/2)*H), w*W, h*H,
                                                  fill=False, edgecolor='r', lw=2))
        plt.title(f'{title} {os.path.basename(p)}'); plt.axis('off'); plt.show()

## 3. RVIP model (Dataset105) — build, export, sanity-check, train

Insertion points come from the IPs-only mask (`IP1 = 1` anterior, `IP2 = 2` inferior —
straight from `dataset.json`); each becomes a small `RVIP_BOX_PX` box. A slice with no
annotated point is simply skipped. Images are the first contrast (`_0000`) exported as
RGB PNGs.

In [ ]:
rvip_out = os.path.join(OUT_ROOT, 'rvip')
rvip_samples = build_train_val(RVIP_IMAGES, RVIP_LABELS, RVIP_SPLITS, FOLD)
print('rvip samples:', len(rvip_samples), '| split:', Counter(s['split'] for s in rvip_samples))

# nifti (_0000) -> RGB png + two insertion-point boxes per slice (IP1=1, IP2=2)
counts_r = yp.export_dataset(rvip_samples, rvip_out, kind='rvip',
                             box_px=RVIP_BOX_PX, channels=CHANNELS,
                             anterior_label=RVIP_ANTERIOR_LABEL,
                             inferior_label=RVIP_INFERIOR_LABEL)
rvip_yaml = yp.write_data_yaml(rvip_out, kind='rvip')
print('wrote', rvip_yaml, counts_r)

In [ ]:
# sanity-check: the two red boxes should sit on the insertion points
show_yolo(os.path.join(rvip_out, 'images/train'), os.path.join(rvip_out, 'labels/train'), title='RVIP')

In [ ]:
rvip_model = yp.train_yolo(rvip_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=RVIP_IMGSZ,
                           project=os.path.join(OUT_ROOT, 'runs'), name='rvip')
print('best weights ->', os.path.join(OUT_ROOT, 'runs', 'rvip', 'weights', 'best.pt'))

## 4. CROP model (Dataset110) — build, export, sanity-check, train

Step-1 whole-heart ROI, the drop-in replacement for the Stage-1 crop nnUNet. Baseline uses
the first contrast only, so `CROP_CHANNELS = (0, 0, 0)` (`_0000` replicated into R/G/B). The
GT box is the squared whole-heart mask (`Heart = 1`).

In [ ]:
crop_out = os.path.join(OUT_ROOT, 'crop')
crop_samples = build_train_val(CROP_IMAGES, CROP_LABELS, CROP_SPLITS, FOLD)
print('crop samples:', len(crop_samples), '| split:', Counter(s['split'] for s in crop_samples))

# single-contrast nifti -> grayscale-as-RGB png + one squared whole-heart box per slice
counts_c = yp.export_dataset(crop_samples, crop_out, kind='crop', channels=CROP_CHANNELS)
crop_yaml = yp.write_data_yaml(crop_out, kind='crop')
print('wrote', crop_yaml, counts_c)

In [ ]:
show_yolo(os.path.join(crop_out, 'images/train'), os.path.join(crop_out, 'labels/train'), title='CROP')

In [ ]:
crop_model = yp.train_yolo(crop_yaml, model=YOLO_WEIGHTS, epochs=EPOCHS, imgsz=CROP_IMGSZ,
                           project=os.path.join(OUT_ROOT, 'runs'), name='crop')
print('best weights ->', os.path.join(OUT_ROOT, 'runs', 'crop', 'weights', 'best.pt'))

## 5. Quick smoke test (optional)
Load the trained weights and run the notebook-facing adapters on one case each, to confirm
they emit the artifacts the pipeline expects: RVIP -> two points (or `None`); CROP -> a
squared box. The `*_from_path` helpers load the SAME `CHANNELS` you trained with.

In [ ]:
# RVIP: first-contrast image -> two points (or None) -> labels 2/3 disk mask
rvip_m = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'rvip', 'weights', 'best.pt'))
pts = yp.detect_rvips_from_path(rvip_m, rvip_samples[0]['image_path'], channels=CHANNELS)
print('detected RVIPs:', pts)

# CROP: single-contrast full image -> square box -> binary crop mask (drop-in for Stage 1)
crop_m = yp.load_model(os.path.join(OUT_ROOT, 'runs', 'crop', 'weights', 'best.pt'))
box = yp.detect_crop_box_from_path(crop_m, crop_samples[0]['image_path'], channels=CROP_CHANNELS)
print('detected crop box (x_min,x_max,y_min,y_max,scale):', box)

## Next step (separate task)
Wiring the two models into `inference.ipynb`:
- replace **cell 1** (crop nnUNet) with `detect_crop_box_from_path(..., channels=CROP_CHANNELS)` -> `crop_box_to_mask`;
- replace **cell 5** (IP nnUNet) with `detect_rvips_from_path(..., channels=CHANNELS)` -> `rvips_to_mask` (labels 2/3);
- keep cells 2/3/6/7 (LV = Dataset100), run analysis in original spacing.

Use the **same `CHANNELS` at inference as at training** — baseline is `(0,0,0)` for both
(first contrast). Low-confidence RVIP detections return `None` (a clean miss, not a wild
outlier) — decide there whether to skip or report a failed point.